## **1. Setup:**

### STRAUSS

First, let's install `strauss`! Just run the code cell below.

*We will use the animation development branch for this notebook. Install can take a while - but you should only need to run it once!*

In [7]:
!pip --quiet install strauss

Import the modules we need...

In [8]:
# strauss imports
from strauss.sonification import Sonification
from strauss.sources import Events, Objects
from strauss import channels
from strauss.score import Score
from strauss.generator import Sampler, Synthesizer, Spectralizer
from strauss import sources as Sources 
import strauss

# other useful modules
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from scipy.signal import savgol_filter
import urllib.request
import os
import zipfile
import glob
import yaml

# modules to display in-notebook
import IPython.display as ipd
from IPython.core.display import display, Markdown, Latex, Image

# set figures to be a decent size by default
import matplotlib
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 18}
matplotlib.rc('font', **font)
matplotlib.rc('figure', **{'figsize':[14.0, 7.0]})

/var/folders/vt/70wqtw0x5zgcdkzdt2kbqsh1g8w3kz/T/ipykernel_43266/2692483155.py:23: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, Markdown, Latex, Image


### SOAR

Then, let's install the module necessary to load SOAR data.

In [9]:
!pip install sunpy[all] sunpy_soar

Import the modules we need...

In [10]:
from sunpy.net import Fido, attrs as a
import cdflib
from sunpy.timeseries import TimeSeries as ts
import sunpy_soar
import numpy as np
import importlib
import sys

# Remove and reload the helper module to get updated version
if 'analysis_helpers' in sys.modules:
    del sys.modules['analysis_helpers']
import analysis_helpers as h

## **2. Getting the Data**

In [ ]:
# Load data directly from SOAR and organize by month
from sunpy.net import Fido, attrs as a
import cdflib

# Define date range
start_date_str = "2025-01-01"
end_date_str = "2025-08-31"

print(f"Fetching SOAR data from {start_date_str} to {end_date_str}...")

# Search for data
instrument = a.Instrument('MAG')
time = a.Time(start_date_str, end_date_str)
level = a.Level(2)
product = a.soar.Product('MAG-RTN-NORMAL-1-MINUTE')

result = Fido.search(time & level & product)
print(f"Found {len(result)} files")

# Download files
files = Fido.fetch(result)
if isinstance(files, str):
    files = [files]

print(f"Downloaded {len(files)} files. Processing...")

# Process and combine data
import pandas as pd
full_data = pd.DataFrame()
for i, file in enumerate(files):
    print(f"  Processing file {i+1}/{len(files)}...", end='\r')
    temp_df = h.cdf2df(file)
    full_data = pd.concat([full_data, temp_df])

full_data.sort_index(inplace=True)
print(f"\nCombined {len(full_data)} data points")

# Group by month
monthly_data = {month: group for month, group in full_data.groupby(full_data.index.month)}
month_names = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 
               'September', 'October', 'November', 'December']
month = [month_names[i-1] for i in sorted(monthly_data.keys())]

print(f"✓ Data organized into {len(monthly_data)} months")


In [ ]:
# Initialize the sonification UI with monthly data
from ui import SonificationUI

if monthly_data:
    app = SonificationUI(monthly_data, month_names)
    app.display()
else:
    print("⚠ No monthly data available. Please run the data loading cell above first.")


/Users/marie-alix.gillyboe/Documents/GitHub/solo8_tutorials/MAG_tutorial/ui.py:235: UserWarning: Discarding nonzero nanoseconds in conversion.
  value=min_date.to_pydatetime().date(),
/Users/marie-alix.gillyboe/Documents/GitHub/solo8_tutorials/MAG_tutorial/ui.py:243: UserWarning: Discarding nonzero nanoseconds in conversion.
  value=max_date.to_pydatetime().date(),


In [ ]:
# Optional: Plot sample data from the loaded data
if monthly_data:
    import matplotlib.pyplot as plt
    
    # Plot first available month
    first_month = list(monthly_data.keys())[0]
    month_to_plot = first_month
    
    fig, axs = plt.subplots(4, 1, sharex=True, figsize=(10, 6))
    
    axs[0].plot(monthly_data[month_to_plot]['|B|'], color='black')
    axs[0].set_ylabel('|B|')
    axs[1].plot(monthly_data[month_to_plot]['BR'], color='red')
    axs[1].set_ylabel('BR')
    axs[2].plot(monthly_data[month_to_plot]['BT'], color='green')
    axs[2].set_ylabel('BT')
    axs[3].plot(monthly_data[month_to_plot]['BN'], color='orange')
    axs[3].set_ylabel('BN')
    
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    
    print(f"Showing data for {month_names[month_to_plot - 1]}")
else:
    print("⚠ No monthly data available. Please run the data loading cell above first.")


## **3. Sonification**

Here we will sonify MAG data as a one-dimensional time series, where some sound property is is varied with time in the sonification, using [STRAUSS](https://strauss.readthedocs.io/en/latest/) python library. 

In strauss we could treat each data point as separate audio Events, with an occurence time mapped from their time occurence.

However, articulating each data point as a separate note for many thousands of data points can require long and drawn-out sonifications.

Here we demonstrate this approach with just out of curiosity. We use the Synthesizer object with the pitch_mapper preset by default - this has a default pitch range of two octaves (a factor of 4 in frequency) and we pick an E3 note (165 Hz) as the base (lowest) frequency.

We will hear the amplitude of the magnetic field of each point as pitch, with their observation mapped to the occurence time.

In [ ]:
# get the |B| data for January
y = monthly_data[1]['|B|'].values.copy()
y = y[~np.isnan(y)]  # Remove NaN values
x = np.arange(len(y))

# plot the data as a reminder with dates on x-axis
plt.scatter(monthly_data[1].index, monthly_data[1]['|B|'], s = 2)
plt.ylabel('|B| (nT)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# specify the base notes used. In this example we use a single E3 note and
# freely vary the pitch via the 'pitch_shift' parameter
notes = [["E3"]]
length = 10  # length of the sonification in seconds
score =  Score(notes, length)
# notes = [["C3","E3","F3","G3","B3","C4","E4","F4","G4","B4","C5","E5","F5","G5","B5"]]   
# score =  Score(notes, 15, pitch_binning="uniform")
    
maps = {'pitch':np.ones(x.size),
        'time': x,
        'pitch_shift':y}
    
# specify audio system (e.g. mono, stereo, 5.1, ...)
system = "mono"
    
# set up synth (this generates the sound using mathematical waveforms)
generator = Synthesizer()
generator.load_preset('pitch_mapper')
    
generator.modify_preset({'note_length':0.1,
                         'volume_envelope': {'use':'on',
                                             # A,D,R values in seconds, S sustain fraction from 0-1 that note
                                             # will 'decay' to (after time A+D)
                                             'A':0.02,    # for such a fast sequence, using ~10 ms values
                                             'D':0.02,    # for such a fast sequence, using ~10 ms values
                                             'S':0.,      # decay to volume 0
                                             'R':0.001}}) # for such a fast sequence, using ~10 ms values
    
# set 0 to 101 percentile limits so the full pitch range is used...
# setting 0 to 101 for pitch means the sonification is 1% longer than the final note position
lims = {'time': ('0%','101%'),
        'pitch_shift': ('0%','101%')}
    
# set up source
sources = Events(maps.keys())
sources.fromdict(maps)
sources.apply_mapping_functions(map_lims=lims)
    
soni = Sonification(score, sources, generator, system)
soni.render()
# soni.save('pitchpm.wav')
dobj = soni.notebook_display(show_waveform=0)

Let's now try the Object approach. In analogy to visual display, the Event representation is like plotting the spectrum as a scatter plot, while an Object representation is like plotting the spectrum as a continuous line.

Let's hear the 1D time-series data sonification, mapping magnetic field to pitch.

In [ ]:
# get the |B| data for January
y = monthly_data[1]['|B|'].values.copy()
y = y[~np.isnan(y)]  # Remove NaN values
x = np.arange(len(y))

# plot the data as a reminder with dates on x-axis
plt.plot(monthly_data[1]['|B|'], linewidth=0.5)
plt.ylabel('|B| (nT)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.title(f'{month[0]}')
plt.show()

notes = [["E3"]]
length = 10  # length of the sonification in seconds
score =  Score(notes, length)
    
# set up synth (this generates the sound using mathematical waveforms)
generator = Synthesizer()
generator.load_preset('pitch_mapper')
    
data = {'pitch':1.,
        'time_evo':x,
        'pitch_shift':y}
    
# set 0 to 101 percentile limits so the full pitch and time range is used...
lims = {'time_evo': ('0%','101%'),
        'pitch_shift': ('0%','101%')}
    
# set up source
sources = Objects(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions(map_lims=lims)
    
soni = Sonification(score, sources, generator, system)
soni.render()
dobj = soni.notebook_display(show_waveform=0)

How about mapping a low-pass filter to magnetic field?

In [ ]:
# get the data for January
y = monthly_data[1]['|B|'].values.copy()
y = y[~np.isnan(y)]  # Remove NaN values
x = np.arange(len(y))

# plot the data as a reminder with dates on x-axis
plt.plot(monthly_data[1]['|B|'], linewidth=0.5)
plt.ylabel('|B| (nT)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.title(f'{month[0]}')
plt.show()

generator = Synthesizer()
generator.modify_preset({'filter':'on'})

# adding a 'textural' sonification using white noise
generator.load_preset('windy')

# we use a (power!) 'chord' here to create more harmonic richness...
notes = [["A2", "E3", 'A3', 'E4']]
score =  Score(notes, 15)

data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'cutoff':[y]*4}

lims = {'time_evo': ('0%','100%'),
        'cutoff': ('0%','100%')}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
plims = {'cutoff': (0.25,0.9)}
sources.apply_mapping_functions(map_lims=lims, param_lims=plims)

soni = Sonification(score, sources, generator, system)
soni.render()
dobj = soni.notebook_display(show_waveform=0)

In fact, there are many expressive properties of sound we could use to represent the data in a similar way. In `strauss` these are referred to as `mappable` properties. 

A subset of these can be used as an evolving property with the `Object` source class. These are referred to as `evolvable` properties. 

Lets show what's available:

In [ ]:
display(Markdown(f"### ***'Mappable'*** properties:"))
for m in Sources.mappable:
  display(Markdown(f' * `{m}` '))

display(Markdown(f"### ***'Evolvable'*** properties:"))
for m in Sources.evolvable:
  display(Markdown(f' * `{m}` '))

We can play around with some of these `evolvable` properties here.

The `idx` variable below controls which evolvable property is selected from the `some_mappings` list. `idx` can be changed to a number from 0 to 5 inclusive to *'index'* a certain sound parameter.

In [ ]:
# A list of some 'evolvable' mappings
some_mappings = ["pitch_shift",
                 "cutoff",
                 "volume",
                 "phi",
                 "volume_lfo/amount", 
                 "volume_lfo/freq_shift",
                 "pitch_lfo/amount", 
                 "pitch_lfo/freq_shift"]

# !! change these (between 0 and 7) to select a different property to map... 
idx = 1

# use a stereo system to allow 'phi' mapping (low pan left and high pan right)
system = "stereo"

display(Markdown(f"### Sonifying {month[0]} data using `evolvable` property - ***`{some_mappings[idx]}`***:"))

# some strauss setup can happen outside the loop...
generator = Synthesizer()
generator.modify_preset({'filter':'on',
                         "pitch_hi":-1, "pitch_lo": 1,
                         "pitch_lfo": {"use": "on", 
                                       "amount":1*("pitch_lfo/freq_shift" in some_mappings[idx]), 
                                       "freq":3*5**("pitch_lfo/freq_shift" not in some_mappings[idx]), 
                                       "phase":0.25},
                         "volume_lfo": {"use": "on", 
                                        "amount":1*("volume_lfo/freq_shift" in some_mappings[idx]), 
                                        "freq":3*5**("volume_lfo/freq_shift" not in some_mappings[idx]), 
                                        "phase":0}
                        })


notes = [["A2","E3","B4","F#4"]]
length = 10
score =  Score(notes, length)

# get the data for January
y = monthly_data[1]['|B|'].values.copy()
y = y[~np.isnan(y)]  # Remove NaN values
x = np.arange(len(y))


data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'cutoff':[0.9]*4,
        'theta':[0.5]*4,
        some_mappings[idx]:[y/y.max()]*4}

# plot the data as a reminder with dates on x-axis
plt.figure(figsize=(10, 5))
plt.plot(monthly_data[1]['|B|'], linewidth=0.5)
plt.ylabel('|B| (nT)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.title('January')
plt.show()

lims = {'time_evo': ('0%','100%'),
        'pitch_shift': ('0%','100%'),
        'cutoff': ('0%','100%'),
        "volume": ('0%','100%'), 
        "phi": (-0.5,1.5),
        "volume_lfo/amount": ('0%','100%'), 
        "volume_lfo/freq_shift": ('0%','100%'),
        "pitch_lfo/amount": ('0%','100%'), 
        "pitch_lfo/freq_shift": ('0%','100%')}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
plims = {'cutoff': (0.25,0.9)}
sources.apply_mapping_functions(map_lims=lims, param_lims=plims)

soni = Sonification(score, sources, generator, system)
soni.render()
dobj = soni.notebook_display(show_waveform=0)

Let's iterate through all months

In [ ]:
# get the data for January
label = ['|B|', 'BR', 'BT', 'BN']
# change this (between 0 and 3) to select a different data to map...
idx_label = 0

# we use a (power!) 'chord' here to create more harmonic richness...
notes = [["A2", "E3", 'A3', 'E4']]
length = 10  # length of the sonification in seconds
score =  Score(notes, length)
    
# set up synth (this generates the sound using mathematical waveforms)
generator = Synthesizer()
generator.modify_preset({'filter':'on'})
# adding a 'textural' sonification using white noise
generator.load_preset('windy')

# iterate over the months to create multiple sonifications
for idx_month in range(len(monthly_data)):
    y = monthly_data[idx_month+1][label[idx_label]].values.copy()
    y = y[~np.isnan(y)]  # Remove NaN values
    x = np.arange(len(y))
    
    # plot the data as a reminder with dates on x-axis
    plt.figure(figsize=(10, 5))
    plt.plot(monthly_data[idx_month+1][label[idx_label]], linewidth=0.5)
    plt.ylabel(f'{label[idx_label]} (nT)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.title(f'{month[idx_month]}')
    plt.show()

    data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'cutoff':[y]*4}
    
    lims = {'time_evo': ('0%','100%'),
            'cutoff': ('0%','100%')}
    
    # set up source
    sources = Objects(data.keys())
    sources.fromdict(data)
    plims = {'cutoff': (0.25,0.9)}
    sources.apply_mapping_functions(map_lims=lims, param_lims=plims)
    soni = Sonification(score, sources, generator, system)
    soni.render()
    dobj = soni.notebook_display(show_waveform=0)

Now, let's try to sonify 2 properties at once!

In [ ]:
# A list of some 'evolvable' mappings
some_mappings = ["pitch_shift",
                 "cutoff",
                 "volume",
                 "phi",
                 "volume_lfo/amount", 
                 "volume_lfo/freq_shift",
                 "pitch_lfo/amount", 
                 "pitch_lfo/freq_shift"]

# !! change these (between 0 and 7) to select a different property to map... 
idx_B = 2
idx_BR = 5

chosen = [some_mappings[idx_B],some_mappings[idx_BR]]

# use a stereo system to allow 'phi' mapping (low pan left and high pan right)
system = "stereo"

# some strauss setup can happen outside the loop...
generator = Synthesizer()

generator = Synthesizer()
# generator.load_preset('windy')
generator.load_preset('pitch_mapper')
generator.modify_preset({'filter':'on',
                         "pitch_hi":-1, "pitch_lo": 1,
                         "pitch_lfo": {"use": "on", 
                                       "amount":1*("pitch_lfo/freq_shift" in chosen), 
                                       "freq":3*5**("pitch_lfo/freq_shift" not in chosen), 
                                       "phase":0.25},
                         "volume_lfo": {"use": "on", 
                                        "amount":1*("volume_lfo/freq_shift" in chosen), 
                                        "freq":3*5**("volume_lfo/freq_shift" not in chosen), 
                                        "phase":0}
                        })


notes = [["A2", "E3"]]
length = 10
score =  Score(notes, length)

lims = {'time_evo': ('0%','100%'),
        'pitch_shift': ('0%','100%'),
        'cutoff': ('0%','100%'),
        "volume": ('0%','100%'), 
        "phi": (-0.5,1.5),
        "volume_lfo/amount": ('0%','100%'), 
        "volume_lfo/freq_shift": ('0%','100%'),
        "pitch_lfo/amount": ('0%','100%'), 
        "pitch_lfo/freq_shift": ('0%','100%')}

display(Markdown(f"## Mapping |B| to ***`{some_mappings[idx_B]}`*** and BR to ***`{some_mappings[idx_BR]}`***:"))

# get data
for idx_month in range(len(monthly_data)):
  display(Markdown(f"### {month[idx_month]}"))
  B = monthly_data[idx_month+1]['|B|'].values.copy()
  B = B[~np.isnan(B)]  # Remove NaN values
  BR = monthly_data[idx_month+1]['BR'].values.copy()
  BR = BR[~np.isnan(BR)]  # Remove NaN values
  time = np.arange(len(B))

  # plot multivariate data ------------------------ 
  plt.figure(figsize=(10, 5))
  plt.plot(monthly_data[idx_month+1]['|B|'], color = 'black', label = '|B|', linewidth=0.5)
  plt.plot(monthly_data[idx_month+1]['BR'], color = 'red', label = 'BR', linewidth=0.5)
  plt.ylabel('(nT)')
  plt.xticks(rotation=45)
  plt.tight_layout()
  plt.legend()
  plt.show()
  # ------------------------------------------------ 

  data = {'pitch':[0,1],
          'time_evo':[time]*2,
          'theta':[0.5]*2,
          some_mappings[idx_B]:[(B - B.min())/(B.max()-B.min())]*2,
          some_mappings[idx_BR]:[(BR - BR.min())/(BR.max()-BR.min())]*2}

  # set up source
  sources = Objects(data.keys())
  sources.fromdict(data)
  plims = {'cutoff': (0.4,1)}
  sources.apply_mapping_functions(map_lims=lims, param_lims=plims)

  soni = Sonification(score, sources, generator, system)
  soni.render()
  dobj = soni.notebook_display(show_waveform=0)

# change back in case cells are run out of order...
generator.load_preset('pitch_mapper')

### Generate the animation

In [ ]:
# Animation features are optional and require additional TTS dependencies
# The core sonification functionality (above) works perfectly without these dependencies
# Uncomment below if you have resolved TTS/NumPy compatibility issues

# !pip install 'strauss[AI-TTS]'

In [ ]:
# Animation rendering is optional and requires TTS dependencies
# This cell also requires a background_video file that may not be available
# The core sonification functionality works without this cell

# Uncomment and run below if you have resolved all TTS/animation dependencies:
'''
# Make frames for animations of mappings as a function of time. This may take several minutes.
import warnings
from pathlib import Path
from strauss.animation import Animate
import shutil
import tempfile

here = Path.cwd()

# Define the final target directory
target_dir_name = Path("figure_animations") / "1D"

# Use a temporary directory for all intermediate files
with tempfile.TemporaryDirectory() as temp_dir_str:
    temp_dir = Path(temp_dir_str)
    print(f"Using temporary directory: {temp_dir}")

    pipe = Animate(temp_dir, pars={"background_video": str(Path(here) / "example_media" / "starfield.mov")})
    pipe.register(f'cutoff', sonification=soni, pre_caption=f'This video shows how cutoff is mapped to the data. Month {month[idx_month]}', post_caption="", stype='animation')
    xp = sources.raw_mapping['time_evo'][0]
    yp = sources.raw_mapping['cutoff'][0]
    nframe = int(soni.score.length*int(pipe.pars['fps']))
    xf = np.linspace(xp[0], xp[-1], nframe)
    yf = np.interp(xf, xp, yp)
    xp, yp = xf, yf
    for i in range(xp.size)[::1]:
        fig, ax1 = plt.subplots()
        ax1.set_xlabel('Time [s]') 
        ax2 = ax1.twinx() 
        ax1.plot(xp, yp)
        ax1.set_ylabel(label[idx_label])
        ax1.tick_params(axis ='y')
        ax1.axvline(xp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ax1.axhline(yp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ymin, ymax = ax1.get_ylim()
        ydel = (ymax-ymin)/(y.max() - y.min())
        yoff = ydel*(y.min()-ymin)/(ymax-ymin)
        ax2.set_ylim(0-yoff, ydel-yoff)
        ax2.set_ylabel('Cutoff')
        ax2.tick_params(axis ='y')
        plt.savefig(pipe.frames['cutoff'].parent / f'frame_{i:05d}.png', dpi=120)
        plt.close()
    print(f"cutoff frames created!")
    pipe.render()

    temp_final_mp4 = temp_dir / "final.mp4" 

    target_dir_name.mkdir(parents=True, exist_ok=True)
    final_target_path = target_dir_name / temp_final_mp4.name
        
    if temp_final_mp4.exists():
        shutil.copy(temp_final_mp4, final_target_path)
        print(f"\nFinal animation copied to: {final_target_path}")
    else:
        warnings.warn(f"Could not find {temp_final_mp4} after rendering.")

from IPython.display import Video
Video(f"figure_animations/1D/final.mp4", embed=True, width=960, height=540)
'''